In [6]:
"""
Análise Micrometeorológica — Balanço de Energia e de Água
Superfícies: Telha Convencional · Grama Esmeralda · Amendoim

Adaptado para dados HOBO — TELHADO_VERDE_27042026_01052026
PPGCLIAMB / INPA / UEA — Tópicos em Meteorologia
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import seaborn as sns

# ══════════════════════════════════════════════════════════════════════
# 1. CARREGAR, PREPARAR E LIMPAR OS DADOS
# ══════════════════════════════════════════════════════════════════════

import os
os.chdir(os.path.dirname(os.path.abspath('Analise.ipynb')))
df = pd.read_csv('TELHADO_VERDE_27042026_01052026.csv', header=1)

# Renomear colunas para nomes curtos e legíveis
df.columns = [
    'idx', 'Data_Hora',
    'vvel',           # Velocidade do vento (m/s)
    'vmax',           # Rajada máxima (m/s)
    'vdir',           # Direção do vento (°)
    'rad',            # Radiação solar incidente (W/m²)
    'temp_telha',     # Temperatura do solo — Convencional (°C × 1000)
    'temp_grama',     # Temperatura do solo — Esmeralda (°C × 1000)
    'temp_amendoim',  # Temperatura do solo — Amendoim (°C × 1000)
    'temp_ar',        # Temperatura do ar (°C × 1000)
    'ur',             # Umidade relativa do ar (% × 1000)
    'prec',           # Precipitação (mm)
    'usoil_grama',    # Umidade volumétrica — Esmeralda (m³/m³)
    'usoil_amendoim', # Umidade volumétrica — Amendoim (m³/m³)
]

# Converter temperaturas (sensor HOBO salva × 1000) e UR
for col in ['temp_telha', 'temp_grama', 'temp_amendoim', 'temp_ar']:
    df[col] = df[col] / 1000
df['ur'] = df['ur'] / 1000

# Parse de data/hora e índice temporal
df['Data_Hora'] = pd.to_datetime(df['Data_Hora'], format='%m/%d/%y %Hh%Mmin%Ss')
df = df.sort_values('Data_Hora').reset_index(drop=True)
df.set_index('Data_Hora', inplace=True)

# --- Limpeza de Outliers (blindagem de sensor / erros pontuais) ---
limites = {'temp_telha': (15, 75), 'temp_grama': (15, 60), 'temp_amendoim': (15, 60), 'temp_ar': (15, 50)}
for col, (lo, hi) in limites.items():
    df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
df.interpolate(method='linear', inplace=True)

# Coluna auxiliar de hora para Seaborn
df['hora'] = df.index.hour

# ══════════════════════════════════════════════════════════════════════
# 2. DADOS DE VAZÃO / DRENAGEM MEDIDOS EM CAMPO
#    (coletados manualmente com proveta em 28/04 e 30/04)
# ══════════════════════════════════════════════════════════════════════

AREA_M2 = 1.21 * 1.17  # m² — área dos protótipos

# Volumes coletados (Litros)
drenagem_campo = {
    # dia       esmeralda  amendoim  telha
    '2026-04-28': (14.000, 14.000, 20.000),
    '2026-04-30': ( 1.946,  1.946,  4.690),
}

def litros_para_mm(vol_L):
    """Converte volume (L) para lâmina equivalente (mm) sobre a área do protótipo."""
    return (vol_L * 0.001 / AREA_M2) * 1000

D = {}
for dia, (ve, va, vc) in drenagem_campo.items():
    D.setdefault('esm', []).append(litros_para_mm(ve))
    D.setdefault('amend', []).append(litros_para_mm(va))
    D.setdefault('conv', []).append(litros_para_mm(vc))

D_esm   = sum(D['esm'])
D_amend = sum(D['amend'])
D_conv  = sum(D['conv'])

# ══════════════════════════════════════════════════════════════════════
# 3. FEATURES DERIVADAS
# ══════════════════════════════════════════════════════════════════════

def pressao_saturacao(T):
    """Pressão de vapor de saturação (kPa) — Magnus."""
    return 0.6108 * np.exp(17.27 * T / (T + 237.3))

df['es'] = pressao_saturacao(df['temp_ar'])
df['ea'] = (df['ur'] / 100) * df['es']
df['dpv_grama']    = pressao_saturacao(df['temp_grama'])    * (1 - df['ur'] / 100)
df['dpv_amendoim'] = pressao_saturacao(df['temp_amendoim']) * (1 - df['ur'] / 100)

# Ilha de calor de superfície: telha vs média das vegetações
df['ilha_calor'] = df['temp_telha'] - (df['temp_grama'] + df['temp_amendoim']) / 2

# Diferença de umidade do solo entre superfícies
df['delta_usoil'] = df['usoil_grama'] - df['usoil_amendoim']

# Ciclo diurno médio
dia_medio = df.groupby('hora').mean(numeric_only=True)

# --- Atraso Térmico (Time Lag) do pico diário ---
lags_minutos = []
for dia, dados_dia in df.groupby(df.index.date):
    pico_telha    = dados_dia['temp_telha'].idxmax()
    pico_amendoim = dados_dia['temp_amendoim'].idxmax()
    if pd.notna(pico_telha) and pd.notna(pico_amendoim):
        lag = (pico_amendoim - pico_telha).total_seconds() / 60
        lags_minutos.append(lag)
atraso_medio = np.mean(lags_minutos)

# --- Ilha de calor: dias secos vs chuvosos ---
chuva_diaria  = df['prec'].resample('D').sum()
dias_chuvosos = chuva_diaria[chuva_diaria > 0].index.date
df['data_apenas'] = df.index.date
ilha_dias_secos = df[~df['data_apenas'].isin(dias_chuvosos)]['ilha_calor'].mean()
ilha_dias_chuva = df[ df['data_apenas'].isin(dias_chuvosos)]['ilha_calor'].mean()

# ══════════════════════════════════════════════════════════════════════
# 4. BALANÇO HÍDRICO (ΔS/Δt = P − ET − R − D)
# ══════════════════════════════════════════════════════════════════════

dz = 130  # mm — espessura do substrato

P_total = df['prec'].sum()  # mm

# ΔS = θ × Δz  →  variação de armazenamento
theta_esm_i   = df['usoil_grama'].iloc[0]
theta_esm_f   = df['usoil_grama'].iloc[-1]
theta_amend_i = df['usoil_amendoim'].iloc[0]
theta_amend_f = df['usoil_amendoim'].iloc[-1]

dS_esm   = (theta_esm_f   - theta_esm_i)   * dz  # mm
dS_amend = (theta_amend_f - theta_amend_i)  * dz  # mm

# Escoamento superficial R assumido nulo (protótipos têm contenção lateral)
R = 0.0

# ET = P − R − D − ΔS
ET_esm   = P_total - R - D_esm   - dS_esm
ET_amend = P_total - R - D_amend - dS_amend

# ══════════════════════════════════════════════════════════════════════
# 5. BALANÇO DE RADIAÇÃO  Rn = ROC↓ − ROC↑ + ROL↓ − ROL↑
# ══════════════════════════════════════════════════════════════════════

sigma    = 5.67e-8  # W m⁻² K⁻⁴
alpha_esm   = 0.62
alpha_amend = 0.58
alpha_conv  = 0.15   # telha de barro típica
eps_veg  = 0.969
eps_conv = 0.90

# Emissividade atmosférica (Brutsaert simplificado)
df['eps_atm'] = 1.24 * (df['ea'] / (df['temp_ar'] + 273.15)) ** (1 / 7)

Tk = lambda col: (df[col] + 273.15)

df['ROC_down']     = df['rad']
df['ROC_up_esm']   = alpha_esm   * df['ROC_down']
df['ROC_up_amend'] = alpha_amend * df['ROC_down']
df['ROC_up_conv']  = alpha_conv  * df['ROC_down']

df['ROL_down']     = df['eps_atm'] * sigma * Tk('temp_ar')**4
df['ROL_up_esm']   = eps_veg  * sigma * Tk('temp_grama')**4
df['ROL_up_amend'] = eps_veg  * sigma * Tk('temp_amendoim')**4
df['ROL_up_conv']  = eps_conv * sigma * Tk('temp_telha')**4

df['Rn_esm']   = df['ROC_down'] - df['ROC_up_esm']   + df['ROL_down'] - df['ROL_up_esm']
df['Rn_amend'] = df['ROC_down'] - df['ROC_up_amend'] + df['ROL_down'] - df['ROL_up_amend']
df['Rn_conv']  = df['ROC_down'] - df['ROC_up_conv']  + df['ROL_down'] - df['ROL_up_conv']

dia_medio = df.groupby('hora').mean(numeric_only=True)

# ══════════════════════════════════════════════════════════════════════
# 6. IMPRESSÃO DOS RESULTADOS
# ══════════════════════════════════════════════════════════════════════

sep = "=" * 60

print(sep)
print("ESTATÍSTICAS GERAIS — VARIÁVEIS METEOROLÓGICAS")
print(sep)
print(df[['temp_telha','temp_grama','temp_amendoim','temp_ar','ur','rad','prec']].describe().round(3))

print(f"\n{sep}")
print("BALANÇO HÍDRICO (período: 27/04 – 01/05/2026)")
print(sep)
print(f"Área dos protótipos      : {AREA_M2:.4f} m²")
print(f"Espessura do substrato   : {dz} mm")
print(f"Precipitação total (P)   : {P_total:.2f} mm")
print()
print(f"{'Componente':<30} {'Esmeralda':>12} {'Amendoim':>12} {'Convencional':>14}")
print("-" * 70)
print(f"{'D – Drenagem (mm)':<30} {D_esm:>12.2f} {D_amend:>12.2f} {D_conv:>14.2f}")
print(f"{'ΔS – Armazenamento (mm)':<30} {dS_esm:>12.2f} {dS_amend:>12.2f} {'—':>14}")
print(f"{'R – Escoa. superficial (mm)':<30} {R:>12.2f} {R:>12.2f} {'—':>14}")
print(f"{'ET – Evapotranspiração (mm)':<30} {ET_esm:>12.2f} {ET_amend:>12.2f} {'—':>14}")

print(f"\n{sep}")
print("BALANÇO DE RADIAÇÃO (médias do período)")
print(sep)
print(f"{'Componente':<30} {'Esmeralda':>12} {'Amendoim':>12} {'Convencional':>14}")
print("-" * 70)
print(f"{'ROC↓ médio (W/m²)':<30} {df['ROC_down'].mean():>12.1f} {df['ROC_down'].mean():>12.1f} {df['ROC_down'].mean():>14.1f}")
print(f"{'ROC↑ médio (W/m²)':<30} {df['ROC_up_esm'].mean():>12.1f} {df['ROC_up_amend'].mean():>12.1f} {df['ROC_up_conv'].mean():>14.1f}")
print(f"{'ROL↓ médio (W/m²)':<30} {df['ROL_down'].mean():>12.1f} {df['ROL_down'].mean():>12.1f} {df['ROL_down'].mean():>14.1f}")
print(f"{'ROL↑ médio (W/m²)':<30} {df['ROL_up_esm'].mean():>12.1f} {df['ROL_up_amend'].mean():>12.1f} {df['ROL_up_conv'].mean():>14.1f}")
print(f"{'Rn médio (W/m²)':<30} {df['Rn_esm'].mean():>12.1f} {df['Rn_amend'].mean():>12.1f} {df['Rn_conv'].mean():>14.1f}")

print(f"\n{sep}")
print("BALANÇO DE ENERGIA — ILHA DE CALOR E INÉRCIA TÉRMICA")
print(sep)
print(f"ΔT médio geral (Telha − Vegetações) : {df['ilha_calor'].mean():.2f} °C")
print(f"ΔT médio — dias secos               : {ilha_dias_secos:.2f} °C")
print(f"ΔT médio — dias com chuva           : {ilha_dias_chuva:.2f} °C")
print(f"ΔT máximo registrado                : {df['ilha_calor'].max():.2f} °C")
print(f"Atraso térmico médio (Amendoim/Telha): {atraso_medio:.0f} min")

# ══════════════════════════════════════════════════════════════════════
# 7. FIGURAS
# ══════════════════════════════════════════════════════════════════════

CORES = {
    'telha':    '#E63946',
    'grama':    '#2D6A4F',
    'amendoim': '#74C69D',
    'ilha':     '#F4A261',
    'dpv':      '#457B9D',
    'prec':     '#1D3557',
    'ur':       '#A8DADC',
    'rad':      '#F4D03F',
    'usoil_g':  '#27AE60',
    'usoil_a':  '#E67E22',
}

horas = dia_medio.index

# Série temporal horária para gráficos temporais
df_h = df.resample('1h').mean(numeric_only=True)

fig = plt.figure(figsize=(20, 28))
fig.patch.set_facecolor('#F8F9FA')
gs = gridspec.GridSpec(5, 2, figure=fig, hspace=0.50, wspace=0.35)

def estilo(ax, titulo, xlabel='Hora do Dia', ylabel=''):
    ax.set_title(titulo, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_facecolor('#FFFFFF')

# ─── 1. Ciclo diurno de temperatura com bandas de confiança ───────────
ax1 = fig.add_subplot(gs[0, :])
sns.lineplot(data=df, x='hora', y='temp_telha',    color=CORES['telha'],    lw=2.5,
             label='Telha Convencional', ax=ax1)
sns.lineplot(data=df, x='hora', y='temp_grama',    color=CORES['grama'],    lw=2.5,
             label='Grama Esmeralda',   ax=ax1)
sns.lineplot(data=df, x='hora', y='temp_amendoim', color=CORES['amendoim'], lw=2.5, ls='--',
             label='Amendoim',          ax=ax1)
sns.lineplot(data=df, x='hora', y='temp_ar',       color='gray',            lw=1.5, ls=':',
             label='Temperatura do ar', ax=ax1)
ax1.set_xticks(range(0, 24))
ax1.legend(loc='upper left', fontsize=9, ncol=4)
estilo(ax1,
       '1. Ciclo Diurno de Temperatura do Solo (Linha: Média | Sombra: IC 95%)',
       ylabel='Temperatura (°C)')

# ─── 2. Ilha de calor de superfície ───────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(horas, dia_medio['ilha_calor'], color=CORES['ilha'], alpha=0.85, edgecolor='white', width=0.8)
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.set_xticks(range(0, 24, 2))
estilo(ax2,
       '2. Ilha de Calor de Superfície\n(Telha − Média das Vegetações)',
       ylabel='ΔT (°C)')

# ─── 3. Umidade relativa — ciclo diurno ───────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(horas, dia_medio['ur'], color=CORES['ur'], lw=2.5, marker='o', ms=4, label='UR do ar')
ax3.fill_between(horas, dia_medio['ur'], alpha=0.2, color=CORES['ur'])
ax3.set_xticks(range(0, 24, 2))
ax3.set_ylim(40, 105)
ax3.legend(fontsize=9)
estilo(ax3,
       '3. Ciclo Diurno de Umidade Relativa do Ar',
       ylabel='UR (%)')

# ─── 4. DPV ───────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
ax4.plot(horas, dia_medio['dpv_grama'],    color=CORES['grama'],    lw=2.2, label='DPV – Esmeralda')
ax4.plot(horas, dia_medio['dpv_amendoim'], color=CORES['amendoim'], lw=2.2, ls='--', label='DPV – Amendoim')
ax4.fill_between(horas, dia_medio['dpv_grama'], dia_medio['dpv_amendoim'],
                 alpha=0.2, color=CORES['amendoim'])
ax4.set_xticks(range(0, 24, 2))
ax4.legend(fontsize=9)
estilo(ax4,
       '4. Déficit de Pressão de Vapor (DPV)\nProxy da demanda evapotranspirativa',
       ylabel='DPV (kPa)')

# ─── 5. Boxplot de temperatura ────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
bp_data = [df['temp_telha'].dropna().values,
           df['temp_grama'].dropna().values,
           df['temp_amendoim'].dropna().values]
bp = ax5.boxplot(bp_data, patch_artist=True, widths=0.5,
                 medianprops=dict(color='white', lw=2))
for patch, color in zip(bp['boxes'], [CORES['telha'], CORES['grama'], CORES['amendoim']]):
    patch.set_facecolor(color); patch.set_alpha(0.85)
for w in bp['whiskers']: w.set(color='gray', lw=1.2)
for c in bp['caps']:     c.set(color='gray', lw=1.2)
for f in bp['fliers']:   f.set(marker='o', markerfacecolor='gray', markersize=2, alpha=0.4)
ax5.set_xticklabels(['Telha\nConvencional', 'Grama\nEsmeralda', 'Amendoim'])
estilo(ax5,
       '5. Distribuição Térmica por Superfície\n(Período completo)',
       xlabel='', ylabel='Temperatura (°C)')

# ─── 6. Umidade volumétrica do solo — série temporal ──────────────────
ax6 = fig.add_subplot(gs[3, 0])
ax6.plot(df_h.index, df_h['usoil_grama'],    color=CORES['usoil_g'], lw=2, label='θ Esmeralda')
ax6.plot(df_h.index, df_h['usoil_amendoim'], color=CORES['usoil_a'], lw=2, ls='--', label='θ Amendoim')
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
ax6.xaxis.set_major_locator(mdates.DayLocator())
ax6.legend(fontsize=9)
estilo(ax6,
       '6. Umidade Volumétrica do Solo (θ)\nSérie temporal',
       xlabel='Data', ylabel='m³/m³')

# ─── 7. Saldo de radiação — ciclo diurno ──────────────────────────────
ax7 = fig.add_subplot(gs[3, 1])
ax7.plot(horas, dia_medio['Rn_esm'],   color=CORES['grama'],    lw=2.2, label='Rn Esmeralda')
ax7.plot(horas, dia_medio['Rn_amend'], color=CORES['amendoim'], lw=2.2, ls='--', label='Rn Amendoim')
ax7.plot(horas, dia_medio['Rn_conv'],  color=CORES['telha'],    lw=2.2, ls=':',  label='Rn Convencional')
ax7.axhline(0, color='gray', lw=0.8, ls='--')
ax7.set_xticks(range(0, 24, 2))
ax7.legend(fontsize=9)
estilo(ax7,
       '7. Saldo de Radiação (Rn)\nCiclo diurno médio',
       ylabel='W/m²')

# ─── 8. Balanço hídrico — barras por protótipo ────────────────────────
ax8 = fig.add_subplot(gs[4, :])
x = np.arange(3)
w = 0.20
labels_surf = ['Esmeralda', 'Amendoim', 'Convencional']
P_vals  = [P_total,  P_total,  P_total]
D_vals  = [D_esm,    D_amend,  D_conv]
dS_vals = [dS_esm,   dS_amend, 0]
ET_vals = [ET_esm,   ET_amend, 0]

ax8.bar(x - 1.5*w, P_vals,  w, color='#3498DB', alpha=0.85, label='P – Precipitação (mm)')
ax8.bar(x - 0.5*w, D_vals,  w, color='#E74C3C', alpha=0.85, label='D – Drenagem (mm)')
ax8.bar(x + 0.5*w, dS_vals, w, color='#F39C12', alpha=0.85, label='ΔS – Armazenamento (mm)')
ax8.bar(x + 1.5*w, ET_vals, w, color='#2ECC71', alpha=0.85, label='ET – Evapotranspiração estimada (mm)')

# Rótulos de valor nas barras
for rect_group in [
    list(zip(x - 1.5*w, P_vals)),
    list(zip(x - 0.5*w, D_vals)),
    list(zip(x + 0.5*w, dS_vals)),
    list(zip(x + 1.5*w, ET_vals)),
]:
    for xi, v in rect_group:
        if v != 0:
            ax8.text(xi, v + 0.5, f'{v:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax8.set_xticks(x)
ax8.set_xticklabels(labels_surf, fontsize=12)
ax8.set_ylabel('mm', fontsize=10)
ax8.legend(fontsize=9, loc='upper right')
ax8.set_facecolor('#FFFFFF')
ax8.grid(True, alpha=0.3, axis='y')
ax8.set_title('8. Balanço Hídrico por Protótipo — Período Completo (27/04 – 01/05/2026)',
              fontsize=11, fontweight='bold', pad=8)

fig.suptitle(
    'Análise Micrometeorológica — Balanço de Energia e de Água\n'
    'Superfícies: Telha Convencional · Grama Esmeralda · Amendoim\n'
    'PPGCLIAMB / INPA / UEA — Tópicos em Meteorologia',
    fontsize=14, fontweight='bold', y=0.995
)

plt.savefig('painel_telhado_verde.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
print("Painel salvo como 'painel_telhado_verde.png'!")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'TELHADO_VERDE_27042026_01052026.csv'

In [8]:
"""
Análise Micrometeorológica — Balanço de Energia e de Água
Superfícies: Telha Convencional · Grama Esmeralda · Amendoim

Adaptado para dados HOBO — TELHADO_VERDE_27042026_01052026
PPGCLIAMB / INPA / UEA — Tópicos em Meteorologia
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import seaborn as sns
import os

# ══════════════════════════════════════════════════════════════════════
# 1. CARREGAR, PREPARAR E LIMPAR OS DADOS
# ══════════════════════════════════════════════════════════════════════

# --- Localizar diretório do notebook de forma robusta ---
try:
    # VS Code com Jupyter: variável especial disponível
    notebook_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    try:
        # JupyterLab / Jupyter clássico
        import IPython
        notebook_dir = IPython.extract_module_locals()[1]['__session__']
        notebook_dir = os.path.dirname(os.path.abspath(notebook_dir))
    except Exception:
        # Fallback: diretório de trabalho atual
        notebook_dir = os.getcwd()

os.chdir(notebook_dir)
print(f"Diretório de trabalho: {notebook_dir}")

# --- Localizar o CSV (aceita .csv ou .xlsx com mesmo nome base) ---
CSV_NAME = 'TELHADO_VERDE_27042026_01052026.csv.xlsx'
XLSX_NAME = 'TELHADO_VERDE_27042026_01052026.xlsx'

csv_path  = os.path.join(notebook_dir, CSV_NAME)
xlsx_path = os.path.join(notebook_dir, XLSX_NAME)

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, header=1)
    print(f"Arquivo carregado: {CSV_NAME}")
elif os.path.exists(xlsx_path):
    df = pd.read_excel(xlsx_path, header=1)
    print(f"Arquivo carregado: {XLSX_NAME}  (xlsx → lido com read_excel)")
else:
    arquivos = os.listdir(notebook_dir)
    raise FileNotFoundError(
        f"Nenhum arquivo de dados encontrado em: {notebook_dir}\n"
        f"Arquivos presentes: {arquivos}\n"
        f"Certifique-se de que '{CSV_NAME}' ou '{XLSX_NAME}' "
        f"está na mesma pasta que o notebook."
    )

# Renomear colunas para nomes curtos e legíveis
df.columns = [
    'idx', 'Data_Hora',
    'vvel',           # Velocidade do vento (m/s)
    'vmax',           # Rajada máxima (m/s)
    'vdir',           # Direção do vento (°)
    'rad',            # Radiação solar incidente (W/m²)
    'temp_telha',     # Temperatura do solo — Convencional (°C × 1000)
    'temp_grama',     # Temperatura do solo — Esmeralda (°C × 1000)
    'temp_amendoim',  # Temperatura do solo — Amendoim (°C × 1000)
    'temp_ar',        # Temperatura do ar (°C × 1000)
    'ur',             # Umidade relativa do ar (% × 1000)
    'prec',           # Precipitação (mm)
    'usoil_grama',    # Umidade volumétrica — Esmeralda (m³/m³)
    'usoil_amendoim', # Umidade volumétrica — Amendoim (m³/m³)
]

# Converter temperaturas (sensor HOBO salva × 1000) e UR
for col in ['temp_telha', 'temp_grama', 'temp_amendoim', 'temp_ar']:
    df[col] = df[col] / 1000
df['ur'] = df['ur'] / 1000

# Parse de data/hora e índice temporal
df['Data_Hora'] = pd.to_datetime(df['Data_Hora'], format='%m/%d/%y %Hh%Mmin%Ss')
df = df.sort_values('Data_Hora').reset_index(drop=True)
df.set_index('Data_Hora', inplace=True)

# --- Limpeza de Outliers (blindagem de sensor / erros pontuais) ---
limites = {
    'temp_telha':    (15, 75),
    'temp_grama':    (15, 60),
    'temp_amendoim': (15, 60),
    'temp_ar':       (15, 50),
}
for col, (lo, hi) in limites.items():
    df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
df.interpolate(method='linear', inplace=True)

# Coluna auxiliar de hora para Seaborn
df['hora'] = df.index.hour

# ══════════════════════════════════════════════════════════════════════
# 2. DADOS DE VAZÃO / DRENAGEM MEDIDOS EM CAMPO
#    (coletados manualmente com proveta em 28/04 e 30/04)
# ══════════════════════════════════════════════════════════════════════

AREA_M2 = 1.21 * 1.17  # m² — área dos protótipos

# Volumes coletados (Litros)
drenagem_campo = {
    # dia           esmeralda  amendoim  telha
    '2026-04-28': (14.000, 14.000, 20.000),
    '2026-04-30': ( 1.946,  1.946,  4.690),
}

def litros_para_mm(vol_L):
    """Converte volume (L) para lâmina equivalente (mm) sobre a área do protótipo."""
    return (vol_L * 0.001 / AREA_M2) * 1000

D = {}
for dia, (ve, va, vc) in drenagem_campo.items():
    D.setdefault('esm',   []).append(litros_para_mm(ve))
    D.setdefault('amend', []).append(litros_para_mm(va))
    D.setdefault('conv',  []).append(litros_para_mm(vc))

D_esm   = sum(D['esm'])
D_amend = sum(D['amend'])
D_conv  = sum(D['conv'])

# ══════════════════════════════════════════════════════════════════════
# 3. FEATURES DERIVADAS
# ══════════════════════════════════════════════════════════════════════

def pressao_saturacao(T):
    """Pressão de vapor de saturação (kPa) — Magnus."""
    return 0.6108 * np.exp(17.27 * T / (T + 237.3))

df['es'] = pressao_saturacao(df['temp_ar'])
df['ea'] = (df['ur'] / 100) * df['es']
df['dpv_grama']    = pressao_saturacao(df['temp_grama'])    * (1 - df['ur'] / 100)
df['dpv_amendoim'] = pressao_saturacao(df['temp_amendoim']) * (1 - df['ur'] / 100)

# Ilha de calor de superfície: telha vs média das vegetações
df['ilha_calor'] = df['temp_telha'] - (df['temp_grama'] + df['temp_amendoim']) / 2

# Diferença de umidade do solo entre superfícies
df['delta_usoil'] = df['usoil_grama'] - df['usoil_amendoim']

# Ciclo diurno médio
dia_medio = df.groupby('hora').mean(numeric_only=True)

# --- Atraso Térmico (Time Lag) do pico diário ---
lags_minutos = []
for dia, dados_dia in df.groupby(df.index.date):
    pico_telha    = dados_dia['temp_telha'].idxmax()
    pico_amendoim = dados_dia['temp_amendoim'].idxmax()
    if pd.notna(pico_telha) and pd.notna(pico_amendoim):
        lag = (pico_amendoim - pico_telha).total_seconds() / 60
        lags_minutos.append(lag)
atraso_medio = np.mean(lags_minutos)

# --- Ilha de calor: dias secos vs chuvosos ---
chuva_diaria  = df['prec'].resample('D').sum()
dias_chuvosos = chuva_diaria[chuva_diaria > 0].index.date
df['data_apenas'] = df.index.date
ilha_dias_secos = df[~df['data_apenas'].isin(dias_chuvosos)]['ilha_calor'].mean()
ilha_dias_chuva = df[ df['data_apenas'].isin(dias_chuvosos)]['ilha_calor'].mean()

# ══════════════════════════════════════════════════════════════════════
# 4. BALANÇO HÍDRICO (ΔS/Δt = P − ET − R − D)
# ══════════════════════════════════════════════════════════════════════

dz = 130  # mm — espessura do substrato

P_total = df['prec'].sum()  # mm

# ΔS = θ × Δz  →  variação de armazenamento
theta_esm_i   = df['usoil_grama'].iloc[0]
theta_esm_f   = df['usoil_grama'].iloc[-1]
theta_amend_i = df['usoil_amendoim'].iloc[0]
theta_amend_f = df['usoil_amendoim'].iloc[-1]

dS_esm   = (theta_esm_f   - theta_esm_i)  * dz  # mm
dS_amend = (theta_amend_f - theta_amend_i) * dz  # mm

# Escoamento superficial R assumido nulo (protótipos têm contenção lateral)
R = 0.0

# ET = P − R − D − ΔS
ET_esm   = P_total - R - D_esm   - dS_esm
ET_amend = P_total - R - D_amend - dS_amend

# ══════════════════════════════════════════════════════════════════════
# 5. BALANÇO DE RADIAÇÃO  Rn = ROC↓ − ROC↑ + ROL↓ − ROL↑
# ══════════════════════════════════════════════════════════════════════

sigma       = 5.67e-8  # W m⁻² K⁻⁴
alpha_esm   = 0.62
alpha_amend = 0.58
alpha_conv  = 0.15     # telha de barro típica
eps_veg     = 0.969
eps_conv    = 0.90

# Emissividade atmosférica (Brutsaert simplificado)
df['eps_atm'] = 1.24 * (df['ea'] / (df['temp_ar'] + 273.15)) ** (1 / 7)

Tk = lambda col: (df[col] + 273.15)

df['ROC_down']     = df['rad']
df['ROC_up_esm']   = alpha_esm   * df['ROC_down']
df['ROC_up_amend'] = alpha_amend * df['ROC_down']
df['ROC_up_conv']  = alpha_conv  * df['ROC_down']

df['ROL_down']     = df['eps_atm'] * sigma * Tk('temp_ar')**4
df['ROL_up_esm']   = eps_veg  * sigma * Tk('temp_grama')**4
df['ROL_up_amend'] = eps_veg  * sigma * Tk('temp_amendoim')**4
df['ROL_up_conv']  = eps_conv * sigma * Tk('temp_telha')**4

df['Rn_esm']   = df['ROC_down'] - df['ROC_up_esm']   + df['ROL_down'] - df['ROL_up_esm']
df['Rn_amend'] = df['ROC_down'] - df['ROC_up_amend'] + df['ROL_down'] - df['ROL_up_amend']
df['Rn_conv']  = df['ROC_down'] - df['ROC_up_conv']  + df['ROL_down'] - df['ROL_up_conv']

dia_medio = df.groupby('hora').mean(numeric_only=True)

# ══════════════════════════════════════════════════════════════════════
# 6. IMPRESSÃO DOS RESULTADOS
# ══════════════════════════════════════════════════════════════════════

sep = "=" * 60

print(sep)
print("ESTATÍSTICAS GERAIS — VARIÁVEIS METEOROLÓGICAS")
print(sep)
print(df[['temp_telha','temp_grama','temp_amendoim','temp_ar','ur','rad','prec']].describe().round(3))

print(f"\n{sep}")
print("BALANÇO HÍDRICO (período: 27/04 – 01/05/2026)")
print(sep)
print(f"Área dos protótipos      : {AREA_M2:.4f} m²")
print(f"Espessura do substrato   : {dz} mm")
print(f"Precipitação total (P)   : {P_total:.2f} mm")
print()
print(f"{'Componente':<30} {'Esmeralda':>12} {'Amendoim':>12} {'Convencional':>14}")
print("-" * 70)
print(f"{'D – Drenagem (mm)':<30} {D_esm:>12.2f} {D_amend:>12.2f} {D_conv:>14.2f}")
print(f"{'ΔS – Armazenamento (mm)':<30} {dS_esm:>12.2f} {dS_amend:>12.2f} {'—':>14}")
print(f"{'R – Escoa. superficial (mm)':<30} {R:>12.2f} {R:>12.2f} {'—':>14}")
print(f"{'ET – Evapotranspiração (mm)':<30} {ET_esm:>12.2f} {ET_amend:>12.2f} {'—':>14}")

print(f"\n{sep}")
print("BALANÇO DE RADIAÇÃO (médias do período)")
print(sep)
print(f"{'Componente':<30} {'Esmeralda':>12} {'Amendoim':>12} {'Convencional':>14}")
print("-" * 70)
print(f"{'ROC↓ médio (W/m²)':<30} {df['ROC_down'].mean():>12.1f} {df['ROC_down'].mean():>12.1f} {df['ROC_down'].mean():>14.1f}")
print(f"{'ROC↑ médio (W/m²)':<30} {df['ROC_up_esm'].mean():>12.1f} {df['ROC_up_amend'].mean():>12.1f} {df['ROC_up_conv'].mean():>14.1f}")
print(f"{'ROL↓ médio (W/m²)':<30} {df['ROL_down'].mean():>12.1f} {df['ROL_down'].mean():>12.1f} {df['ROL_down'].mean():>14.1f}")
print(f"{'ROL↑ médio (W/m²)':<30} {df['ROL_up_esm'].mean():>12.1f} {df['ROL_up_amend'].mean():>12.1f} {df['ROL_up_conv'].mean():>14.1f}")
print(f"{'Rn médio (W/m²)':<30} {df['Rn_esm'].mean():>12.1f} {df['Rn_amend'].mean():>12.1f} {df['Rn_conv'].mean():>14.1f}")

print(f"\n{sep}")
print("BALANÇO DE ENERGIA — ILHA DE CALOR E INÉRCIA TÉRMICA")
print(sep)
print(f"ΔT médio geral (Telha − Vegetações) : {df['ilha_calor'].mean():.2f} °C")
print(f"ΔT médio — dias secos               : {ilha_dias_secos:.2f} °C")
print(f"ΔT médio — dias com chuva           : {ilha_dias_chuva:.2f} °C")
print(f"ΔT máximo registrado                : {df['ilha_calor'].max():.2f} °C")
print(f"Atraso térmico médio (Amendoim/Telha): {atraso_medio:.0f} min")

# ══════════════════════════════════════════════════════════════════════
# 7. FIGURAS
# ══════════════════════════════════════════════════════════════════════

CORES = {
    'telha':    '#E63946',
    'grama':    '#2D6A4F',
    'amendoim': '#74C69D',
    'ilha':     '#F4A261',
    'dpv':      '#457B9D',
    'prec':     '#1D3557',
    'ur':       '#A8DADC',
    'rad':      '#F4D03F',
    'usoil_g':  '#27AE60',
    'usoil_a':  '#E67E22',
}

horas = dia_medio.index

# Série temporal horária para gráficos temporais
df_h = df.resample('1h').mean(numeric_only=True)

fig = plt.figure(figsize=(20, 28))
fig.patch.set_facecolor('#F8F9FA')
gs = gridspec.GridSpec(5, 2, figure=fig, hspace=0.50, wspace=0.35)

def estilo(ax, titulo, xlabel='Hora do Dia', ylabel=''):
    ax.set_title(titulo, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_facecolor('#FFFFFF')

# ─── 1. Ciclo diurno de temperatura com bandas de confiança ───────────
ax1 = fig.add_subplot(gs[0, :])
sns.lineplot(data=df, x='hora', y='temp_telha',    color=CORES['telha'],    lw=2.5,
             label='Telha Convencional', ax=ax1)
sns.lineplot(data=df, x='hora', y='temp_grama',    color=CORES['grama'],    lw=2.5,
             label='Grama Esmeralda',   ax=ax1)
sns.lineplot(data=df, x='hora', y='temp_amendoim', color=CORES['amendoim'], lw=2.5, ls='--',
             label='Amendoim',          ax=ax1)
sns.lineplot(data=df, x='hora', y='temp_ar',       color='gray',            lw=1.5, ls=':',
             label='Temperatura do ar', ax=ax1)
ax1.set_xticks(range(0, 24))
ax1.legend(loc='upper left', fontsize=9, ncol=4)
estilo(ax1,
       '1. Ciclo Diurno de Temperatura do Solo (Linha: Média | Sombra: IC 95%)',
       ylabel='Temperatura (°C)')

# ─── 2. Ilha de calor de superfície ───────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(horas, dia_medio['ilha_calor'], color=CORES['ilha'], alpha=0.85, edgecolor='white', width=0.8)
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.set_xticks(range(0, 24, 2))
estilo(ax2,
       '2. Ilha de Calor de Superfície\n(Telha − Média das Vegetações)',
       ylabel='ΔT (°C)')

# ─── 3. Umidade relativa — ciclo diurno ───────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(horas, dia_medio['ur'], color=CORES['ur'], lw=2.5, marker='o', ms=4, label='UR do ar')
ax3.fill_between(horas, dia_medio['ur'], alpha=0.2, color=CORES['ur'])
ax3.set_xticks(range(0, 24, 2))
ax3.set_ylim(40, 105)
ax3.legend(fontsize=9)
estilo(ax3,
       '3. Ciclo Diurno de Umidade Relativa do Ar',
       ylabel='UR (%)')

# ─── 4. DPV ───────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
ax4.plot(horas, dia_medio['dpv_grama'],    color=CORES['grama'],    lw=2.2, label='DPV – Esmeralda')
ax4.plot(horas, dia_medio['dpv_amendoim'], color=CORES['amendoim'], lw=2.2, ls='--', label='DPV – Amendoim')
ax4.fill_between(horas, dia_medio['dpv_grama'], dia_medio['dpv_amendoim'],
                 alpha=0.2, color=CORES['amendoim'])
ax4.set_xticks(range(0, 24, 2))
ax4.legend(fontsize=9)
estilo(ax4,
       '4. Déficit de Pressão de Vapor (DPV)\nProxy da demanda evapotranspirativa',
       ylabel='DPV (kPa)')

# ─── 5. Boxplot de temperatura ────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
bp_data = [df['temp_telha'].dropna().values,
           df['temp_grama'].dropna().values,
           df['temp_amendoim'].dropna().values]
bp = ax5.boxplot(bp_data, patch_artist=True, widths=0.5,
                 medianprops=dict(color='white', lw=2))
for patch, color in zip(bp['boxes'], [CORES['telha'], CORES['grama'], CORES['amendoim']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
for w in bp['whiskers']: w.set(color='gray', lw=1.2)
for c in bp['caps']:     c.set(color='gray', lw=1.2)
for f in bp['fliers']:   f.set(marker='o', markerfacecolor='gray', markersize=2, alpha=0.4)
ax5.set_xticklabels(['Telha\nConvencional', 'Grama\nEsmeralda', 'Amendoim'])
estilo(ax5,
       '5. Distribuição Térmica por Superfície\n(Período completo)',
       xlabel='', ylabel='Temperatura (°C)')

# ─── 6. Umidade volumétrica do solo — série temporal ──────────────────
ax6 = fig.add_subplot(gs[3, 0])
ax6.plot(df_h.index, df_h['usoil_grama'],    color=CORES['usoil_g'], lw=2, label='θ Esmeralda')
ax6.plot(df_h.index, df_h['usoil_amendoim'], color=CORES['usoil_a'], lw=2, ls='--', label='θ Amendoim')
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
ax6.xaxis.set_major_locator(mdates.DayLocator())
ax6.legend(fontsize=9)
estilo(ax6,
       '6. Umidade Volumétrica do Solo (θ)\nSérie temporal',
       xlabel='Data', ylabel='m³/m³')

# ─── 7. Saldo de radiação — ciclo diurno ──────────────────────────────
ax7 = fig.add_subplot(gs[3, 1])
ax7.plot(horas, dia_medio['Rn_esm'],   color=CORES['grama'],    lw=2.2, label='Rn Esmeralda')
ax7.plot(horas, dia_medio['Rn_amend'], color=CORES['amendoim'], lw=2.2, ls='--', label='Rn Amendoim')
ax7.plot(horas, dia_medio['Rn_conv'],  color=CORES['telha'],    lw=2.2, ls=':',  label='Rn Convencional')
ax7.axhline(0, color='gray', lw=0.8, ls='--')
ax7.set_xticks(range(0, 24, 2))
ax7.legend(fontsize=9)
estilo(ax7,
       '7. Saldo de Radiação (Rn)\nCiclo diurno médio',
       ylabel='W/m²')

# ─── 8. Balanço hídrico — barras por protótipo ────────────────────────
ax8 = fig.add_subplot(gs[4, :])
x = np.arange(3)
w = 0.20
labels_surf = ['Esmeralda', 'Amendoim', 'Convencional']
P_vals  = [P_total,  P_total,  P_total]
D_vals  = [D_esm,    D_amend,  D_conv]
dS_vals = [dS_esm,   dS_amend, 0]
ET_vals = [ET_esm,   ET_amend, 0]

ax8.bar(x - 1.5*w, P_vals,  w, color='#3498DB', alpha=0.85, label='P – Precipitação (mm)')
ax8.bar(x - 0.5*w, D_vals,  w, color='#E74C3C', alpha=0.85, label='D – Drenagem (mm)')
ax8.bar(x + 0.5*w, dS_vals, w, color='#F39C12', alpha=0.85, label='ΔS – Armazenamento (mm)')
ax8.bar(x + 1.5*w, ET_vals, w, color='#2ECC71', alpha=0.85, label='ET – Evapotranspiração estimada (mm)')

# Rótulos de valor nas barras
for rect_group in [
    list(zip(x - 1.5*w, P_vals)),
    list(zip(x - 0.5*w, D_vals)),
    list(zip(x + 0.5*w, dS_vals)),
    list(zip(x + 1.5*w, ET_vals)),
]:
    for xi, v in rect_group:
        if v != 0:
            ax8.text(xi, v + 0.5, f'{v:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax8.set_xticks(x)
ax8.set_xticklabels(labels_surf, fontsize=12)
ax8.set_ylabel('mm', fontsize=10)
ax8.legend(fontsize=9, loc='upper right')
ax8.set_facecolor('#FFFFFF')
ax8.grid(True, alpha=0.3, axis='y')
ax8.set_title('8. Balanço Hídrico por Protótipo — Período Completo (27/04 – 01/05/2026)',
              fontsize=11, fontweight='bold', pad=8)

fig.suptitle(
    'Análise Micrometeorológica — Balanço de Energia e de Água\n'
    'Superfícies: Telha Convencional · Grama Esmeralda · Amendoim\n'
    'PPGCLIAMB / INPA / UEA — Tópicos em Meteorologia',
    fontsize=14, fontweight='bold', y=0.995
)

plt.savefig('painel_telhado_verde.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
print("Painel salvo como 'painel_telhado_verde.png'!")
plt.show()

Diretório de trabalho: c:\Users\alima\OneDrive\Documentos\Meus projetos\Analise do MicroClima de Manaus


UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 15-16: invalid continuation byte